# Reading from bronze table

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

In [0]:
df.display()

# data transformations

In [0]:
df.schema.fields

In [0]:
for instance in df.schema.fields:
    print(instance.name)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

## Trimming

In [0]:
for instance in df.schema.fields:
    if instance.dataType.typeName() == "string":
        df = df.withColumn(instance.name, F.trim(df[instance.name]))

## Normalization

In [0]:
df = df.withColumn(
    "cst_marital_status",
    F.when(F.upper(F.col("cst_marital_status")) == "M", "Married")
     .when(F.upper(F.col("cst_marital_status")) == "S", "Single")
     .otherwise("Unknown")
).withColumn(
    "cst_gndr",
    F.when(F.upper(F.col("cst_gndr")) == "M", "Male")
     .when(F.upper(F.col("cst_gndr")) == "F", "Female")
     .otherwise("n/a")
)

## Renaming the columns

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.display()

# write into siver table

In [0]:
df.write \
 . mode("overwrite") \
 . format("delta") \
  .saveAsTable("workspace.silver.crm_customers")